# Lesson 22: Word embeddings — Word2Vec and GloVe 🧭
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Understand the idea **"you shall know a word by the company it keeps"**
2. Generate Word2Vec training pairs by hand, then train a real **Word2Vec** model
3. Explore pretrained **GloVe** vectors: similar words, analogies, a 2-D map
4. Fix last lesson's search engine with embeddings, and spot **bias** in embeddings

**⏱️ Timing:** idea 10' | Word2Vec 20' | GloVe 20' | search & bias 10'

> 💡 No GPU needed.

## 👥 An analogy: you are who you hang out with
If you've never heard the word *"wug"* but read: *"The wug barked at the postman and wagged its tail"* — you can guess a wug is like a dog.
Words that appear in **similar contexts** have **similar meanings**. Word embeddings turn this into numbers: each word gets a vector (say 100 numbers), and words used in similar ways end up **close together**.

| | One-hot / Bag of Words (Lesson 21) | Word embeddings |
|---|---|---|
| "cat" vs "kitten" | completely different columns | very close vectors |
| Size | one column per word (huge) | ~50–300 numbers |
| Learned? | no, just counting | yes, from billions of words |

- **Word2Vec** (Google, 2013): a tiny neural network learns vectors by predicting **neighbouring words**
- **GloVe** (Stanford, 2014): learns vectors from a big table of **how often words appear together**

In [ ]:
!pip -q install gensim
import gensim, numpy as np, matplotlib.pyplot as plt
print("gensim", gensim.__version__)

## Part 1: How Word2Vec builds its training data 🔧
**Skip-gram**: for each word, predict the words in a small **window** around it.

In [ ]:
sentence = "the quick brown fox jumps over the lazy dog".split()
window = 2
pairs = []
for i, centre in enumerate(sentence):
    for j in range(max(0, i - window), min(len(sentence), i + window + 1)):
        if j != i:
            pairs.append((centre, sentence[j]))
print(len(pairs), "training pairs, e.g.:")
for p in pairs[:10]: print("  given", p[0], "→ predict", p[1])

The network gets **fake tasks** like "given *fox*, predict *jumps*". We don't care about the predictions — we keep the **weights it learns** along the way. Those weights are the word vectors!

## Part 2: Train your own Word2Vec 🏋️
We'll use the **Brown corpus** (1 million words of American English from 1961, built into NLTK).

In [ ]:
import nltk
nltk.download("brown", quiet=True)
from nltk.corpus import brown
from gensim.models import Word2Vec

sentences = [[w.lower() for w in s if w.isalpha()] for s in brown.sents()]
print(len(sentences), "sentences, e.g.", sentences[0][:12])

w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=5, sg=1, epochs=5, workers=2)
print("vocabulary size:", len(w2v.wv))
print("vector for 'school' (first 8 numbers):", w2v.wv["school"][:8].round(2))

In [ ]:
for word in ["school", "money", "monday", "happy", "car"]:
    print(f"{word:7} →", [w for w, _ in w2v.wv.most_similar(word, topn=6)])

### 🧩 Exercise 1
1. Try 5 words of your own. Which neighbours make sense? Which are odd?
2. The Brown corpus is from 1961. Try `computer`, `phone`, `internet`. What happens, and why?
3. Change `window=5` to `window=2`, retrain and compare `most_similar("happy")`. (Small windows → similar *grammar*; big windows → similar *topic*.)

## Part 3: Pretrained GloVe vectors 🌍
Training on 1 million words is small. GloVe was trained on **6 billion words** (Wikipedia + news). Download: about 130 MB.

In [ ]:
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-100")     # 400,000 words × 100 numbers
print("king ≈", glove["king"][:6].round(2), "...")
for word in ["kiwi", "python", "apple"]:
    print(f"{word:7} →", [w for w, _ in glove.most_similar(word, topn=6)])

Notice **kiwi** (bird? fruit? New Zealander?), **python** and **apple**: each word gets only **one** vector, so all its meanings get mixed together. That's a key limitation (BERT fixes it next lesson).

### ✨ Word arithmetic: king − man + woman ≈ ?

In [ ]:
def analogy(a, b, c):
    """a is to b as c is to ?"""
    return [w for w, _ in glove.most_similar(positive=[b, c], negative=[a], topn=3)]

print("man : king   :: woman : ", analogy("man", "king", "woman"))
print("france : paris :: japan : ", analogy("france", "paris", "japan"))
print("walk : walked :: swim : ", analogy("walk", "walked", "swim"))
print("big : bigger  :: small : ", analogy("big", "bigger", "small"))

In [ ]:
# Draw a 2-D map of 100-D vectors (PCA squashes the dimensions, keeping as much spread as possible)
from sklearn.decomposition import PCA
words = ["cat", "dog", "horse", "cow", "apple", "banana", "orange", "grape",
         "paris", "london", "tokyo", "auckland", "red", "blue", "green", "yellow",
         "run", "walk", "swim", "jump"]
xy = PCA(n_components=2).fit_transform(np.array([glove[w] for w in words]))
plt.figure(figsize=(8, 6)); plt.scatter(xy[:, 0], xy[:, 1])
for (x, y), w in zip(xy, words): plt.annotate(w, (x, y), fontsize=11)
plt.title("GloVe words squashed into 2-D"); plt.show()

### 🧩 Exercise 2
1. Invent 3 analogies of your own. Which work? Which fail?
2. Add 4 new words to the map (e.g. sports). Do they form their own cluster?
3. `glove.doesnt_match(["breakfast", "lunch", "dinner", "car"])` — try a few odd-one-out puzzles.

## Part 4: Smarter search with embeddings 🔍
Last lesson, *"I misplaced my coat"* couldn't find the lost-jacket notice. Let's average the word vectors of each sentence and compare those instead.

In [ ]:
notices = [
    "The library will be closed on Monday for stocktaking. Books can be returned to the office.",
    "Basketball trials for the senior team are on Wednesday after school in the gym.",
    "The canteen has a new menu: sushi on Tuesdays and pasta on Thursdays.",
    "Lost property: a blue jacket and a calculator were found in the gym.",
    "Reminder: the swimming sports day is next Friday. Bring your togs and a towel.",
]
import re
def sentence_vector(text):
    words = [w for w in re.findall(r"[a-z]+", text.lower()) if w in glove and w not in {"the", "a", "my", "i", "and", "on", "in"}]
    return np.mean([glove[w] for w in words], axis=0)

def cos(u, v): return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

def search(query):
    q = sentence_vector(query)
    for s, n in sorted([(cos(q, sentence_vector(n)), n) for n in notices], reverse=True)[:2]:
        print(f"{s:.2f}  {n}")

search("I misplaced my coat")
print()
search("what food can I eat at lunch")

## ⚠️ Part 5: Embeddings learn our biases too
Embeddings learn from human writing, so they also learn **stereotypes** (like the BERT experiment in Lesson 14).

In [ ]:
for job in ["nurse", "engineer", "doctor", "receptionist", "programmer", "teacher"]:
    lean = cos(glove[job], glove["he"]) - cos(glove[job], glove["she"])
    print(f"{job:13} {'he ←' if lean > 0 else '     '} {lean:+.3f} {'→ she' if lean < 0 else ''}")

**Discuss:** if a hiring tool used these vectors to match CVs to jobs, what could go wrong? Researchers have developed "debiasing" methods, but they only partly work.

### 🚀 Challenge
Use `glove.most_similar(positive=[...], negative=[...])` to explore one more relationship (country → currency, animal → baby animal…). Write up your best and worst result.

## 📌 Summary
- **Embedding** = a list of numbers that captures a word's meaning from its context
- **Word2Vec**: predict neighbours (skip-gram / CBOW); **GloVe**: learn from co-occurrence counts
- Similar words are close; relationships become **directions** (king − man + woman ≈ queen)
- Limits: **one vector per word** (kiwi 🥝 = kiwi 🐦) and inherited **bias**